> VIGOR 2.12.1 example. Install with `pip install "VIGOR[io,progress,plot] @ git+https://github.com/sanchestm/VIGOR"` (see the [README](../../../README.md)). Outputs are cleared: this notebook was checked against the 2.12.1 API but not re-executed for this release.

# HS project 06 — LOCO mvGWAS to canonical xarray/Zarr

In [ ]:
from pathlib import Path
import sys, os, json, warnings
import numpy as np
import pandas as pd
import xarray as xr

# Find ROOT/vigor_run/project_config.py regardless of notebook cwd.
def _find_run(start=None):
    p=Path(os.environ.get('VIGOR_PROJECT_ROOT', os.environ.get('DOMINO_PROJECT_ROOT', Path.cwd()))) if start is None else Path(start)
    for q in [p,*p.parents]:
        cand=q/'vigor_run'/'project_config.py'
        if cand.exists(): return cand.parent
        if q.name=='vigor_run' and (q/'project_config.py').exists(): return q
    raise FileNotFoundError('Set VIGOR_PROJECT_ROOT or place vigor_run under the project root.')
RUN=_find_run(); sys.path.insert(0,str(RUN))
from project_config import *
import vigor
print('Project root:',ROOT)
print('VIGOR:',vigor.__version__)
assert tuple(map(int, vigor.__version__.split('.')[:3])) >= (2, 12, 1), vigor.__version__

In [ ]:
train=pd.read_csv(TRAIN_CSV,index_col=0);train.index=train.index.astype(str);train.columns=train.columns.str.replace('regressedlr_','',regex=False)
GWAS_STORE.parent.mkdir(parents=True,exist_ok=True)
# Dense canonical result written directly to Zarr. This may be long-running but is restartable at the store/cache layer.
gwas=vigor.mvGWAS(bfile=PLINK_PREFIX,pheno=train,traits=list(train.columns),model='add-dom',out=GWAS_STORE,engine='pipeline',block_size=8192,trait_mode='independent',decomposition='exact',compute_dtype='float64',n_jobs=BLAS_THREADS,return_results=False,progress=True)
print(gwas)

In [ ]:
# xarray is the normal analysis/plotting handle.
trait=str(gwas.trait.values[0])
view=gwas.sel(trait=trait)
hits=view.where((view.neglog_p>=5).compute(),drop=True)
print(trait,hits.sizes)
# Optional explicit long export:
vigor.export_gwas_parquet(gwas,PARQUET/'mvgwas_hits.parquet',metric='neglog_p',threshold=5.0)

In [ ]:
try:
    import hvplot.xarray
    display(view.neglog_p.hvplot.scatter(x='position',by='chromosome',width=1100,height=380,title=f'{trait}: additive GWAS'))
except Exception as exc:
    warnings.warn(str(exc))

Variant metadata are stored once along the variant dimension. Filtering traits/chromosomes/statistics therefore avoids the key duplication of a long `variant × trait` DataFrame.